In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy.stats import skew, kurtosis
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem, DataStructs
from itertools import combinations
from itertools import combinations
from collections import Counter
from mordred import Calculator, descriptors
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
import pubchempy as pcp
%config InlineBackend.figure_format = 'retina'
%matplotlib inline
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
plt.rcParams['font.sans-serif'] = ["Arial"]

In [ ]:
# 读取 CSV 文件
file_name = "../../data/source_data/property_models/DN.csv"  # installed external data
df = pd.read_csv(file_name)

# 确保 SMILES 列存在
smiles_column = "SMILES"
if smiles_column not in df.columns:
    raise ValueError(f"CSV 文件中未找到 '{smiles_column}' 列，请检查列名")

# 提取 SMILES，并去除空值
smiles_list = df[smiles_column].dropna().tolist()

# 初始化 Mordred 计算器
calc = Calculator(descriptors, ignore_3D=True)

# 获取描述符名称（直接通过map提取）
descriptor_names = list(map(lambda x: x, calc.descriptors))  # 这里直接提取元组中的名称

# 计算描述符（带进度条）
results = []
valid_smiles = []
for smiles in tqdm(smiles_list, desc="计算 Mordred 描述符", unit="分子"):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        descriptors_values = calc(mol)
        results.append(descriptors_values)
        valid_smiles.append(smiles)  # 记录有效的 SMILES

# 转换为 DataFrame
df_descriptors = pd.DataFrame(results)

# 将列名设置为描述符的名称
df_descriptors.columns = descriptor_names

# 将原始的 SMILES 列插入到数据框的第一列
df_descriptors.insert(0, "SMILES", valid_smiles)

# 保存为新的 CSV 文件
# output_file = "mordred_descriptors.csv"
# df_descriptors.to_csv(output_file, index=False)

# print(f"\n计算完成！结果已保存到 {output_file}")

In [ ]:
# 将所有列转换为数值，无法转换为数值的会被标记为 NaN
df_numeric = df_descriptors.apply(pd.to_numeric, errors='coerce')

# 将所有 NaN 填充为 0
df_numeric.fillna(0, inplace=True)

df_numeric['SMILES']=smiles_list

# 打印处理后的数据形状
print(f"最终处理后的数据形状: {df_numeric.shape}")

# 保存结果为新的 CSV 文件
# output_file = "filled_mordred_descriptors.csv"
# df_numeric.to_csv(output_file, index=False)

# print(f"\n计算完成！结果已保存到 {output_file}")

In [ ]:
# 删除全为0的列之前的列数
initial_columns = df_numeric.shape[1]

# 删除全为0的列
df_numeric = df_numeric.loc[:, (df_numeric != 0).any(axis=0)]

# 删除后的列数
final_columns = df_numeric.shape[1]

# 打印删除的列数
print(f"删除了 {initial_columns - final_columns} 列")

In [ ]:
# 计算相关性矩阵（忽略第一列 SMILES）
correlation_matrix = df_numeric.iloc[:, 1:].corr()

# 找出相关性大于0.8的列对
high_corr_var = set()
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if abs(correlation_matrix.iloc[i, j]) > 0.8:
            colname = correlation_matrix.columns[i]
            high_corr_var.add(colname)

# 删除高相关性的列（只保留一个）
df_filtered = df_numeric.drop(columns=high_corr_var)

# 计算最终相关性矩阵
final_corr_matrix = df_filtered.iloc[:, 1:].corr()

# 打印筛选后的数据形状
print(f"筛选后的数据形状: {df_filtered.shape}")

# 保存最终的 DataFrame 和相关性矩阵
# df_filtered.to_csv("df_filtered.csv", index=False)
# final_corr_matrix.to_csv("correlation_matrix_filtered.csv")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import matplotlib.ticker as ticker

# 设置阈值
threshold = 0.025

# 选择需要归一化的列（忽略第一列 SMILES）
descriptors = df_filtered.iloc[:, 1:]

# MinMax 归一化
scaler = MinMaxScaler()
descriptors_normalized = scaler.fit_transform(descriptors)

# 将归一化后的数据转换为 DataFrame，保持列名
descriptors_normalized_df = pd.DataFrame(descriptors_normalized, columns=descriptors.columns)

# 计算每个描述符的方差
variances = descriptors_normalized_df.var()

# ---------------- 绘图部分 ----------------
plt.figure(figsize=(7.5, 6))
plt.hist(variances, bins=30, color='moccasin', edgecolor='black')
# plt.title('Variance Distribution of Descriptors', fontsize=20)
plt.xlabel('Standard deviation', fontsize=24)
plt.ylabel('Counts', fontsize=24)
plt.grid(True, linestyle="--", linewidth=0.6, alpha=0.7)
plt.tick_params(axis="both", direction="in", labelsize=20)
plt.axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold = {threshold}')
plt.legend(fontsize=16)

# 设置 x 轴刻度格式为两位小数
ax = plt.gca()
ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))

# 加粗边框
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
    spine.set_color('black')

plt.tight_layout()
plt.show()

# ---------------- 筛选方差高于阈值的描述符 ----------------
high_variance_columns = variances[variances >= threshold].index.tolist()

# 构建新的 DataFrame，保留 SMILES 和筛选后的描述符列
df_final = pd.concat([df_filtered.iloc[:, [0]], df_filtered[high_variance_columns]], axis=1)
df_final.to_csv("df_final.csv", index=False)

# 可选：打印信息
print(f"原始描述符数量：{descriptors.shape[1]}")
print(f"筛选后保留的描述符数量：{len(high_variance_columns)}")

In [ ]:
variances

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# 假设 variances 是你已计算的包含标准差的 Series，筛选出大于 0.025 的描述符
high_variance = variances[variances > 0.025]

# 排序和重置索引
sorted_data = pd.DataFrame({'Descriptors': high_variance.index, 'std_dev': high_variance.values})
sorted_data = sorted_data.sort_values(by='std_dev', ascending=False).reset_index(drop=True)

# 颜色映射
colors = cm.plasma(sorted_data.index / len(sorted_data))

# 绘图
plt.figure(figsize=(12, 8))
fontsize = 20
bars = plt.bar(sorted_data.index, sorted_data['std_dev'], color=colors)

# 字体和标签设置
plt.rc('xtick', labelsize=fontsize)
plt.rc('ytick', labelsize=fontsize)
plt.rcParams['font.family'] = "Arial"
plt.xlabel('Descriptor', fontsize=fontsize)
plt.ylabel('Standard deviation', fontsize=fontsize)
plt.tick_params(direction='in')

# 设置 y 轴刻度和范围
plt.ylim(0, 0.15)
plt.yticks(np.arange(0, 0.18, 0.04))

# 设置 x 轴刻度
x_ticks = sorted_data['Descriptors'][::5]
x_tick_positions = sorted_data.index[::5]
# plt.xticks(x_tick_positions, x_ticks, rotation=45, ha='right', fontsize=12)

# 减少左右边距
plt.xlim(-1.5, len(sorted_data) - 0.5)

# 加粗边框
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)

plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# 假设 variances 是你已计算的包含标准差的 Series，筛选出大于 0.025 的描述符
high_variance = variances[variances > 0.025]

# 排序和重置索引
sorted_data = pd.DataFrame({'Descriptors': high_variance.index, 'std_dev': high_variance.values})
sorted_data = sorted_data.sort_values(by='std_dev', ascending=False).reset_index(drop=True)

# 颜色映射
colors = cm.plasma(sorted_data.index / len(sorted_data))

# 绘图
plt.figure(figsize=(12, 8))
fontsize = 20
bars = plt.bar(sorted_data.index, sorted_data['std_dev'], color=colors)

# 字体和标签设置
plt.rc('xtick', labelsize=fontsize)
plt.rc('ytick', labelsize=fontsize)
plt.rcParams['font.family'] = "Arial"
plt.xlabel('Descriptor', fontsize=fontsize)
plt.ylabel('Standard deviation', fontsize=fontsize)
plt.tick_params(direction='in')

# 设置 y 轴刻度和范围
plt.ylim(0, 0.15)
plt.yticks(np.arange(0, 0.18, 0.04))

# 使用提供的 xticks 描述符
x_ticks = [
    "MAXsCH3", "AXp-6d", "JGI3", "PEOE_VSA2", "EState_VSA3", 
    "MATS2s", "SlogP_VSA4", "PEOE_VSA10", "GATS3c", "SIC1", 
    "GATS2c", "AATSC3v"
]

# 对应的 x_tick_positions 为索引位置
x_tick_positions = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110]

# 设置 x 轴刻度
plt.xticks(x_tick_positions, x_ticks, rotation=45, ha='right', fontsize=12)

# 减少左右边距
plt.xlim(-1.5, len(sorted_data) - 0.5)

# 加粗边框
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)

plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# 假设 variances 是你已计算的包含标准差的 Series，筛选出大于 0.025 的描述符
high_variance = variances[variances > 0.025]

# 排序和重置索引
sorted_data = pd.DataFrame({'Descriptors': high_variance.index, 'std_dev': high_variance.values})
sorted_data = sorted_data.sort_values(by='std_dev', ascending=False).reset_index(drop=True)

# 颜色映射
colors = cm.plasma(sorted_data.index / len(sorted_data))

# 绘图
plt.figure(figsize=(12, 8))
fontsize = 20
bars = plt.bar(sorted_data.index, sorted_data['std_dev'], color=colors)

# 字体和标签设置
plt.rc('xtick', labelsize=fontsize)
plt.rc('ytick', labelsize=fontsize)
plt.rcParams['font.family'] = "Arial"
plt.xlabel('Descriptor', fontsize=fontsize)
plt.ylabel('Standard deviation', fontsize=fontsize)
plt.tick_params(direction='in')

# 设置 y 轴刻度和范围
plt.ylim(0, 0.15)
plt.yticks(np.arange(0, 0.18, 0.04))

# 使用提供的 xticks 描述符
x_ticks = [
    "MAXsCH3", "nN", "AXp-6d", "AXp-5d", "JGI3", "AATS7d", 
    "PEOE_VSA2", "NssO", "EState_VSA3", "ATSC0c", "MATS2s", 
    "AATS4dv", "SlogP_VSA4", "nBondsT", "PEOE_VSA10", "ETA_beta_ns_d", 
    "GATS3c", "AATSC1i", "SIC1", "ATSC2dv", "GATS2c", "MDEC-12", 
    "AATSC3v", "GATS1v"
]

# 对应的 x_tick_positions 为索引位置
x_tick_positions = [
    0, 5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 
    85, 90, 95, 100, 105, 110, 115
]

# 设置 x 轴刻度
plt.xticks(x_tick_positions, x_ticks, rotation=45, ha='right', fontsize=12)

# 减少左右边距
plt.xlim(-1.5, len(sorted_data) - 0.5)

# 加粗边框
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)

plt.show()


In [ ]:
x_ticks

In [ ]:
x_tick_positions

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 读取 CSV 文件
file_name = "../../data/source_data/property_models/DN.csv"  # installed external data
df = pd.read_csv(file_name)

# 确保 DM 列存在
dn_column = "DN"
if dn_column not in df.columns:
    raise ValueError(f"CSV 文件中未找到 '{dm_column}' 列，请检查列名")

# 绘图：美化版的 DM 分布直方图
plt.figure(figsize=(7.5, 6))
sns.histplot(df[dn_column], kde=True, color='#EB8763', bins=30)

# 设置标题和轴标签样式
# plt.title('DN Distribution', fontsize=20)
plt.xlabel('Donor Number (kcal/mol)', fontsize=24)
plt.ylabel('Counts', fontsize=24)
plt.grid(True, linestyle="--", linewidth=0.6, alpha=0.7)
plt.tick_params(axis="both", direction="in", labelsize=20)
plt.xlim(-1, 62)

# 加粗边框，设置为黑色
for spine in plt.gca().spines.values():
    spine.set_linewidth(2)
    spine.set_color('black')

# 显示图形
plt.tight_layout()
plt.show()

# 输出 DN 的统计指标
mean_val = df[dn_column].mean()
var_val = df[dn_column].var()
std_val = df[dn_column].std()

print(f"DN 平均值（Mean）: {mean_val:.2f}")
print(f"DN 方差（Variance）: {var_val:.2f}")
print(f"DN 标准差（Standard Deviation）: {std_val:.2f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns

# 模拟数据
np.random.seed(42)
n_samples = 100
df = pd.DataFrame({
    "PCE": np.random.uniform(10, 22, n_samples),             
    "Cost": np.random.uniform(0.5, 3.0, n_samples),           
    "Dipole": np.random.uniform(0.5, 5.0, n_samples),         
    "DN": np.random.uniform(10, 30, n_samples)                
})

# 统一方向（全部转为最小化）
df_obj = df.copy()
df_obj["PCE"] = -df_obj["PCE"]
df_obj["Dipole"] = -df_obj["Dipole"]
df_obj["DN"] = -df_obj["DN"]

# 提取帕累托前沿
def is_pareto_efficient(costs):
    is_efficient = np.ones(costs.shape[0], dtype=bool)
    for i, c in enumerate(costs):
        if is_efficient[i]:
            is_efficient[is_efficient] = np.any(costs[is_efficient] < c, axis=1) | np.all(costs[is_efficient] == c, axis=1)
            is_efficient[i] = True
    return is_efficient

cost_array = df_obj.values
is_pareto = is_pareto_efficient(cost_array)
pareto_df = df[is_pareto]

# 添加标签
df["Pareto"] = ["Front" if x else "Others" for x in is_pareto]

# --------------------------------------
# ✅ 1. 两两散点图（pairplot）
# --------------------------------------
sns.pairplot(df, vars=["PCE", "Cost", "Dipole", "DN"], hue="Pareto", palette={"Front": "red", "Others": "gray"})
plt.suptitle("Pairwise Scatter Plot with Pareto Front", y=1.02)
plt.tight_layout()
plt.show()

# --------------------------------------
# ✅ 2. 三维图（PCE, Cost, Dipole）
# --------------------------------------
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')
colors = ['red' if p == "Front" else 'gray' for p in df["Pareto"]]

ax.scatter(df["PCE"], df["Cost"], df["Dipole"], c=colors, s=50, alpha=0.8)
ax.set_xlabel("PCE")
ax.set_ylabel("Cost")
ax.set_zlabel("Dipole")
ax.set_title("3D Pareto Front (PCE-Cost-Dipole)")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 读取CSV文件
file_path = 'DN_Pred.csv'
df = pd.read_csv(file_path)

# 获取Num_Features列
num_features = df['Num_Features']

# 设置统一的 Y 轴范围
y_range_rmse = [0, 11]  # RMSE的Y轴范围
y_range_spearman = [0.75, 1]  # Spearman的Y轴范围

# 创建子图
fig, axes = plt.subplots(4, 1, figsize=(12, 10))

# 模型名称
models = ['Ridge Regression', 'Random Forest', 'Gradient Boosting', 'CatBoost']

# 遍历四个模型进行绘制
for i, model in enumerate(models):
    # 获取每个模型对应的列
    spearman_col = df.iloc[:, 1 + 4 * i]  # Spearman系数列
    spearman_std_col = df.iloc[:, 2 + 4 * i]  # Spearman标准差列
    rmse_col = df.iloc[:, 3 + 4 * i]  # RMSE列
    rmse_std_col = df.iloc[:, 4 + 4 * i]  # RMSE标准差列
    
    # 设置柱状图的位置
    bar_width = 0.35
    index = np.arange(len(num_features))
    
    # 绘制RMSE柱状图（左边y轴）
    ax1 = axes[i]
    ax1.bar(index, rmse_col, bar_width, label='RMSE', yerr=rmse_std_col, capsize=5, color='#66B89F', edgecolor='black')
    ax1.set_ylabel('RMSE', fontsize=14)
    ax1.set_ylim(y_range_rmse)  # 设置RMSE的Y轴范围
    ax1.tick_params(axis="y", direction="in", labelsize=12)
    ax1.set_xticks(index + bar_width / 2)
    ax1.set_xticklabels(num_features,  ha='right', fontsize=12)
    
    # 创建右边y轴共享左边y轴
    ax2 = ax1.twinx()
    ax2.bar(index + bar_width, spearman_col, bar_width, label='Spearman', yerr=spearman_std_col, capsize=5, color='#8A97C0', edgecolor='black')
    ax2.set_ylabel('Spearman', fontsize=14)
    ax2.set_ylim(y_range_spearman)  # 设置Spearman的Y轴范围
    ax2.tick_params(axis="y", direction="in", labelsize=12)
    
    # 设置网格
    ax1.grid(True, linestyle="--", linewidth=0.6, alpha=0.7)

    # 添加图例
    ax1.legend(loc='upper left', fontsize=12)
    ax2.legend(loc='upper right', fontsize=12)

    # 加粗边框
    for spine in ax1.spines.values():
        spine.set_linewidth(2)
    for spine in ax2.spines.values():
        spine.set_linewidth(2)

# 调整子图间距
plt.tight_layout()

# 显示图形
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 读取CSV文件
file_path = 'DN_Pred.csv'
df = pd.read_csv(file_path)

# 获取Num_Features列
num_features = df['Num_Features']

# 设置统一的 Y 轴范围
y_range_rmse = [4, 11]  # RMSE的Y轴范围
y_range_spearman = [0.7, 0.98]  # Spearman的Y轴范围

# 创建子图
fig, axes = plt.subplots(4, 1, figsize=(12, 9))

# 模型名称
models = ['Ridge Regression', 'Random Forest', 'Gradient Boosting', 'CatBoost']

# 遍历四个模型进行绘制
for i, model in enumerate(models):
    # 获取每个模型对应的列
    spearman_col = df.iloc[:, 1 + 4 * i]  # Spearman系数列
    spearman_std_col = df.iloc[:, 2 + 4 * i]  # Spearman标准差列
    rmse_col = df.iloc[:, 3 + 4 * i]  # RMSE列
    rmse_std_col = df.iloc[:, 4 + 4 * i]  # RMSE标准差列
    
    # 设置柱状图的位置
    bar_width = 0.25
    index = np.arange(len(num_features))
    
    # 绘制RMSE柱状图（左边y轴）
    ax1 = axes[i]
    ax1.bar(index, rmse_col, bar_width, label='RMSE', yerr=rmse_std_col, capsize=5, color='#66B89F', edgecolor='none')
    ax1.set_ylabel('RMSE', fontsize=18)
    ax1.set_ylim(y_range_rmse)  # 设置RMSE的Y轴范围
    ax1.tick_params(axis="y", direction="in", labelsize=16)
    
    # 创建右边y轴共享左边y轴
    ax2 = ax1.twinx()
    ax2.bar(index + bar_width, spearman_col, bar_width, label='Spearman', yerr=spearman_std_col, capsize=5, color='#8A97C0', edgecolor='none')
    ax2.set_ylabel('Spearman', fontsize=18)
    ax2.set_ylim(y_range_spearman)  # 设置Spearman的Y轴范围
    ax2.tick_params(axis="y", direction="in", labelsize=16)
    
    # 设置网格
    ax1.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)

    # 仅在最后一个子图添加图例
    if i == 3:
        ax1.legend(loc='upper left', fontsize=14)
        ax2.legend(loc='upper right', fontsize=14)

    # 加粗边框
    for spine in ax1.spines.values():
        spine.set_linewidth(2)
    for spine in ax2.spines.values():
        spine.set_linewidth(2)

# 统一设置最后一个子图的x轴
axes[-1].set_xticks(index)
axes[-1].set_xticklabels(num_features, ha='right', fontsize=16)

# 仅最后一个子图添加x轴标签
axes[-1].set_xlabel('Number of Features', fontsize=18)

# 去掉前三个子图的x轴ticks
for i in range(3):
    axes[i].set_xticks([])

# 调整子图间距
plt.tight_layout()

# 显示图形
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 读取CSV文件
file_path = 'DN_Pred.csv'
df = pd.read_csv(file_path)

# 获取Num_Features列
num_features = df['Num_Features']

# 设置统一的 Y 轴范围
y_range_rmse = [5, 11]  # RMSE的Y轴范围
y_range_spearman = [0.7, 0.98]  # Spearman的Y轴范围

# 创建子图
fig, axes = plt.subplots(4, 1, figsize=(11, 8))

# 模型名称
models = ['Ridge Regression', 'Random Forest', 'Gradient Boosting', 'CatBoost']

# 遍历四个模型进行绘制
for i, model in enumerate(models):
    # 获取每个模型对应的列
    spearman_col = df.iloc[:, 1 + 4 * i]  # Spearman系数列
    spearman_std_col = df.iloc[:, 2 + 4 * i]  # Spearman标准差列
    rmse_col = df.iloc[:, 3 + 4 * i]  # RMSE列
    rmse_std_col = df.iloc[:, 4 + 4 * i]  # RMSE标准差列
    
    # 设置柱状图的位置
    bar_width = 0.2
    index = np.arange(len(num_features))
    
    # 绘制RMSE柱状图（左边y轴）
    ax1 = axes[i]
    ax1.bar(index, rmse_col, bar_width, label='RMSE', yerr=rmse_std_col, capsize=5, color='#66B89F', edgecolor='none')
    ax1.set_ylabel('RMSE', fontsize=20)
    ax1.set_ylim(y_range_rmse)  # 设置RMSE的Y轴范围
    ax1.tick_params(axis="y", direction="in", labelsize=18)
    
    # 创建右边y轴共享左边y轴
    ax2 = ax1.twinx()
    ax2.bar(index + bar_width, spearman_col, bar_width, label='Spearman', yerr=spearman_std_col, capsize=3, color='#8A97C0', edgecolor='none')
    ax2.set_ylabel('Spearman', fontsize=20)
    ax2.set_ylim(y_range_spearman)  # 设置Spearman的Y轴范围
    ax2.tick_params(axis="y", direction="in", labelsize=18)
    
    # 设置网格
    ax1.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)

    # 仅在最后一个子图添加图例
    if i == 3:
        ax1.legend(loc='upper left', fontsize=16)
        ax2.legend(loc='upper right', fontsize=16)

    # 加粗边框
    for spine in ax1.spines.values():
        spine.set_linewidth(2)
    for spine in ax2.spines.values():
        spine.set_linewidth(2)

# 统一设置最后一个子图的x轴
axes[-1].set_xticks(index)
axes[-1].set_xticklabels(num_features, rotation=25, fontsize=18)

# 仅最后一个子图添加x轴标签
axes[-1].set_xlabel('Number of Features', fontsize=20)

# 去掉前三个子图的x轴ticks
for i in range(3):
    axes[i].set_xticks([])

# 修改最后一个子图的x轴ticks朝里
axes[-1].tick_params(axis='x', direction='in', labelsize=18)

# 调整子图间距
plt.tight_layout()

# 显示图形
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 读取CSV文件
file_path = 'DN_Pred.csv'
df = pd.read_csv(file_path)

# 获取Num_Features列
num_features = df['Num_Features']

# 设置统一的 Y 轴范围
y_range_rmse = [5, 9]  # RMSE的Y轴范围
y_range_spearman = [0.7, 0.95]  # Spearman的Y轴范围

# 创建子图
fig, axes = plt.subplots(4, 1, figsize=(11, 8))

# 模型名称
models = ['Ridge Regression', 'Random Forest', 'Gradient Boosting', 'CatBoost']

# 遍历四个模型进行绘制
for i, model in enumerate(models):
    # 获取每个模型对应的列
    spearman_col = df.iloc[:, 1 + 4 * i]  # Spearman系数列
    rmse_col = df.iloc[:, 3 + 4 * i]  # RMSE列
    
    # 设置柱状图的位置
    bar_width = 0.2
    index = np.arange(len(num_features))
    
    # 绘制RMSE柱状图（左边y轴）
    ax1 = axes[i]
    ax1.bar(index, rmse_col, bar_width, label='RMSE', color='#66B89F', edgecolor='none')
    ax1.set_ylabel('RMSE', fontsize=20)
    ax1.set_ylim(y_range_rmse)  # 设置RMSE的Y轴范围
    ax1.tick_params(axis="y", direction="in", labelsize=18)
    
    # 创建右边y轴共享左边y轴
    ax2 = ax1.twinx()
    ax2.bar(index + bar_width, spearman_col, bar_width, label='Spearman', color='#8A97C0', edgecolor='none')
    ax2.set_ylabel('Spearman', fontsize=20)
    ax2.set_ylim(y_range_spearman)  # 设置Spearman的Y轴范围
    ax2.tick_params(axis="y", direction="in", labelsize=18)
    
    # 设置网格
    ax1.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)

    # 仅在最后一个子图添加图例
    if i == 3:
        ax1.legend(loc='upper left', fontsize=16)
        ax2.legend(loc='upper right', fontsize=16)

    # 加粗边框
    for spine in ax1.spines.values():
        spine.set_linewidth(2)
    for spine in ax2.spines.values():
        spine.set_linewidth(2)

# 统一设置最后一个子图的x轴
axes[-1].set_xticks(index)
axes[-1].set_xticklabels(num_features, rotation=25, fontsize=18)

# 仅最后一个子图添加x轴标签
axes[-1].set_xlabel('Number of Features', fontsize=20)

# 去掉前三个子图的x轴ticks
for i in range(3):
    axes[i].set_xticks([])

# 修改最后一个子图的x轴ticks朝里
axes[-1].tick_params(axis='x', direction='in', labelsize=18)

# 调整子图间距
plt.tight_layout()

# 显示图形
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 读取CSV文件
file_path = 'DN_Pred.csv'
df = pd.read_csv(file_path)

# 获取Num_Features列
num_features = df['Num_Features']

# 设置统一的 Y 轴范围
y_range_rmse = [5, 9]  # RMSE的Y轴范围
y_range_spearman = [0.7, 0.95]  # Spearman的Y轴范围

# 创建子图
fig, axes = plt.subplots(4, 1, figsize=(11, 8))

# 模型名称
models = ['Ridge Regression', 'Random Forest', 'Gradient Boosting', 'CatBoost']

# 遍历四个模型进行绘制
for i, model in enumerate(models):
    # 获取每个模型对应的列
    spearman_col = df.iloc[:, 1 + 4 * i]  # Spearman系数列
    rmse_col = df.iloc[:, 3 + 4 * i]  # RMSE列
    
    # 设置柱状图的位置
    bar_width = 0.2
    index = np.arange(len(num_features))
    
    # 绘制RMSE柱状图（左边y轴）
    ax1 = axes[i]
    ax1.bar(index, rmse_col, bar_width, label='RMSE', color='#66B89F', edgecolor='none')
    ax1.set_ylabel('RMSE', fontsize=20)
    ax1.set_ylim(y_range_rmse)  # 设置RMSE的Y轴范围
    ax1.set_yticks([6, 7, 8])  # 设置Y轴刻度为6, 7, 8
    ax1.tick_params(axis="y", direction="in", labelsize=18)
    
    # 创建右边y轴共享左边y轴
    ax2 = ax1.twinx()
    ax2.bar(index + bar_width, spearman_col, bar_width, label='Spearman', color='#8A97C0', edgecolor='none')
    ax2.set_ylabel('Spearman', fontsize=20)
    ax2.set_ylim(y_range_spearman)  # 设置Spearman的Y轴范围
    ax2.tick_params(axis="y", direction="in", labelsize=18)
    
    # 设置网格
    ax1.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)

    # 仅在最后一个子图添加图例
    if i == 3:
        ax1.legend(loc='upper left', fontsize=16)
        ax2.legend(loc='upper right', fontsize=16)

    # 加粗边框
    for spine in ax1.spines.values():
        spine.set_linewidth(2)
    for spine in ax2.spines.values():
        spine.set_linewidth(2)

# 统一设置最后一个子图的x轴
axes[-1].set_xticks(index)
axes[-1].set_xticklabels(num_features, rotation=25, fontsize=18)

# 仅最后一个子图添加x轴标签
axes[-1].set_xlabel('Number of Features', fontsize=20)

# 去掉前三个子图的x轴ticks
for i in range(3):
    axes[i].set_xticks([])

# 修改最后一个子图的x轴ticks朝里
axes[-1].tick_params(axis='x', direction='in', labelsize=18)

# 调整子图间距
plt.tight_layout()

# 显示图形
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 读取CSV文件
file_path = 'DM_Pred.csv'
df = pd.read_csv(file_path)

# 获取Num_Features列
num_features = df['Num_Features']

# 设置统一的 Y 轴范围
y_range_rmse = [0.7, 1.2]  # RMSE的Y轴范围
y_range_spearman = [0.6, 0.95]  # Spearman的Y轴范围

# 创建子图
fig, axes = plt.subplots(4, 1, figsize=(11, 8))

# 模型名称
models = ['Ridge Regression', 'Random Forest', 'Gradient Boosting', 'CatBoost']

# 遍历四个模型进行绘制
for i, model in enumerate(models):
    # 获取每个模型对应的列
    spearman_col = df.iloc[:, 1 + 4 * i]  # Spearman系数列
    spearman_std_col = df.iloc[:, 2 + 4 * i]  # Spearman标准差列
    rmse_col = df.iloc[:, 3 + 4 * i]  # RMSE列
    rmse_std_col = df.iloc[:, 4 + 4 * i]  # RMSE标准差列
    
    # 设置柱状图的位置
    bar_width = 0.2
    index = np.arange(len(num_features))
    
    # 绘制RMSE柱状图（左边y轴）
    ax1 = axes[i]
    ax1.bar(index, rmse_col, bar_width, label='RMSE', yerr=rmse_std_col, capsize=5, color='#66B89F', edgecolor='none')
    ax1.set_ylabel('RMSE', fontsize=20)
    ax1.set_ylim(y_range_rmse)  # 设置RMSE的Y轴范围
    ax1.tick_params(axis="y", direction="in", labelsize=18)
    
    # 创建右边y轴共享左边y轴
    ax2 = ax1.twinx()
    ax2.bar(index + bar_width, spearman_col, bar_width, label='Spearman', yerr=spearman_std_col, capsize=3, color='#8A97C0', edgecolor='none')
    ax2.set_ylabel('Spearman', fontsize=20)
    ax2.set_ylim(y_range_spearman)  # 设置Spearman的Y轴范围
    ax2.tick_params(axis="y", direction="in", labelsize=18)
    
    # 设置网格
    ax1.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)

    # 仅在最后一个子图添加图例
    if i == 3:
        ax1.legend(loc='upper left', fontsize=16)
        ax2.legend(loc='upper right', fontsize=16)

    # 加粗边框
    for spine in ax1.spines.values():
        spine.set_linewidth(2)
    for spine in ax2.spines.values():
        spine.set_linewidth(2)

# 统一设置最后一个子图的x轴
axes[-1].set_xticks(index)
axes[-1].set_xticklabels(num_features, rotation=25,fontsize=18)

# 仅最后一个子图添加x轴标签
axes[-1].set_xlabel('Number of Features', fontsize=20)

# 去掉前三个子图的x轴ticks
for i in range(3):
    axes[i].set_xticks([])

# 修改最后一个子图的x轴ticks朝里
axes[-1].tick_params(axis='x', direction='in', labelsize=18)

# 调整子图间距
plt.tight_layout()

# 显示图形
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 读取CSV文件
file_path = 'DM_Pred.csv'
df = pd.read_csv(file_path)

# 获取Num_Features列
num_features = df['Num_Features']

# 设置统一的 Y 轴范围
y_range_rmse = [0.7, 1.12]  # RMSE的Y轴范围
y_range_spearman = [0.6, 0.88]  # Spearman的Y轴范围

# 创建子图
fig, axes = plt.subplots(4, 1, figsize=(11, 8))

# 模型名称
models = ['Ridge Regression', 'Random Forest', 'Gradient Boosting', 'CatBoost']

# 遍历四个模型进行绘制
for i, model in enumerate(models):
    # 获取每个模型对应的列
    spearman_col = df.iloc[:, 1 + 4 * i]  # Spearman系数列
    rmse_col = df.iloc[:, 3 + 4 * i]  # RMSE列
    
    # 设置柱状图的位置
    bar_width = 0.2
    index = np.arange(len(num_features))
    
    # 绘制RMSE柱状图（左边y轴）
    ax1 = axes[i]
    ax1.bar(index, rmse_col, bar_width, label='RMSE', color='#66B89F', edgecolor='none')
    ax1.set_ylabel('RMSE', fontsize=20)
    ax1.set_ylim(y_range_rmse)  # 设置RMSE的Y轴范围
    ax1.set_yticks([0.8, 0.9, 1.0,1.1])
    ax1.tick_params(axis="y", direction="in", labelsize=18)
    
    # 创建右边y轴共享左边y轴
    ax2 = ax1.twinx()
    ax2.bar(index + bar_width, spearman_col, bar_width, label='Spearman', color='#8A97C0', edgecolor='none')
    ax2.set_ylabel('Spearman', fontsize=20)
    ax2.set_ylim(y_range_spearman)  # 设置Spearman的Y轴范围
    ax2.set_yticks([0.6, 0.7, 0.8])
    ax2.tick_params(axis="y", direction="in", labelsize=18)
    
    # 设置网格
    ax1.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)

    # 仅在最后一个子图添加图例
    if i == 3:
        ax1.legend(loc='upper left', fontsize=16)
        ax2.legend(loc='upper right', fontsize=16)

    # 加粗边框
    for spine in ax1.spines.values():
        spine.set_linewidth(2)
    for spine in ax2.spines.values():
        spine.set_linewidth(2)

# 统一设置最后一个子图的x轴
axes[-1].set_xticks(index)
axes[-1].set_xticklabels(num_features, rotation=25, fontsize=18)

# 仅最后一个子图添加x轴标签
axes[-1].set_xlabel('Number of Features', fontsize=20)

# 去掉前三个子图的x轴ticks
for i in range(3):
    axes[i].set_xticks([])

# 修改最后一个子图的x轴ticks朝里
axes[-1].tick_params(axis='x', direction='in', labelsize=18)

# 调整子图间距
plt.tight_layout()

# 显示图形
plt.show()